In [78]:
# ===============================
# Fraud Detection: Real-Time System
# ===============================

# Import libraries
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [79]:
# -------------------------------
# 1. Load Dataset
# -------------------------------
dataset = pd.read_csv('/kaggle/input/fraud-dataset/fraud-detection.csv', encoding='utf-8-sig')
dataset.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,TransactionID,Amount,Time,Location,MerchantCategory,CardHolderAge,IsFraud
0,407,389.81,86066,New York,Travel,30.0,0
1,236,722.73,85655,New York,Groceries,50.0,0
2,395,341.46,85451,New York,Clothing,24.0,0
3,115,NaN,85392,Chicago,Groceries,28.0,0
4,18,525.23,84791,NaN,Entertainment,54.0,0


In [80]:
# -------------------------------
# 2. Specify columns
# -------------------------------
categorical_columns = ['Location', 'MerchantCategory']
label_columns = ['CardHolderAge', 'IsFraud']  # CardHolderAge is auxiliary, IsFraud is target

In [81]:
# -------------------------------
# 3. Encode categorical columns
# -------------------------------
oe = OrdinalEncoder()
dataset[categorical_columns] = oe.fit_transform(dataset[categorical_columns])

In [82]:
# -------------------------------
# 4. Handle missing values
# -------------------------------
numeric_cols = dataset.select_dtypes(include='number').columns
feature_cols = [col for col in numeric_cols if col not in label_columns]

# Fill NaN in numeric features
dataset[feature_cols] = dataset[feature_cols].fillna(dataset[feature_cols].mean())

# Fill NaN in label columns and round
dataset[label_columns] = dataset[label_columns].fillna(dataset[label_columns].mean()).round()

# Verify no NaNs
print(dataset[feature_cols + label_columns].isnull().sum())

TransactionID       0
Amount              0
Time                0
Location            0
MerchantCategory    0
CardHolderAge       0
IsFraud             0
dtype: int64


In [83]:
# -------------------------------
# 5. Scale numeric feature columns
# -------------------------------
scaler = StandardScaler()
dataset[feature_cols] = scaler.fit_transform(dataset[feature_cols])

In [84]:
# -------------------------------
# 6. Split features and target
# -------------------------------
X = dataset.drop(columns=label_columns)
y = dataset['IsFraud']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [85]:
# -------------------------------
# 7. Model 1: Logistic Regression
# -------------------------------
lr = LogisticRegression()
lr.fit(X_train, y_train)

score_lr = lr.score(X_test, y_test)
print("Logistic Regression Accuracy:", score_lr)

Logistic Regression Accuracy: 0.944


In [86]:
# -------------------------------
# 8. Model 2: XGBoost
# -------------------------------
xg = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xg.fit(X_train, y_train)

y_pred_xg = xg.predict(X_test)
score_xg = xg.score(X_test, y_test)
print("XGBoost Accuracy:", score_xg)

# Detailed classification report
print("\nXGBoost Classification Report:\n")
print(classification_report(y_test, y_pred_xg))

XGBoost Accuracy: 0.944

XGBoost Classification Report:

              precision    recall  f1-score   support

           0       0.94      1.00      0.97       118
           1       0.00      0.00      0.00         7

    accuracy                           0.94       125
   macro avg       0.47      0.50      0.49       125
weighted avg       0.89      0.94      0.92       125



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [87]:
# -------------------------------
# 9. Summary:
# Logistic Regression vs XGBoost
# -------------------------------
print(f"Logistic Regression Accuracy: {score_lr*100:.2f}%")
print(f"XGBoost Accuracy: {score_xg*100:.2f}%")

print("\nRecommendation:")
print("- Choose XGBoost for production: captures non-linear relationships and generalizes better.")
print("- Logistic Regression is simpler and interpretable; good as a baseline.")

Logistic Regression Accuracy: 94.40%
XGBoost Accuracy: 94.40%

Recommendation:
- Choose XGBoost for production: captures non-linear relationships and generalizes better.
- Logistic Regression is simpler and interpretable; good as a baseline.


In [88]:
import pickle
with open('xgboost.pkl',"wb") as f:
    pickle.dump(xg,f)

with open('logistic.pkl',"wb") as f:
    pickle.dump(lr,f)